# Atividade: CNNs para Classificação

Neste notebook, iremos preparar nosso próprio dataset e treinar um modelo de classificação de imagens.

## Preparando os dados

Os dados desta atividade serão baixados da internet. Utilizaremos para isso buscadores comuns. Em seguida, dividiremos em treinamento e validação.

In [21]:
#%pip install icrawler
%pip install pyimagedl


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: C:\Users\neope\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [22]:
import os
import shutil
import random
#from icrawler.builtin import GoogleImageCrawler, BingImageCrawler
from imagedl import imagedl

### Adquirindo as Imagens

Utilizaremos o iCrawler para baixar imagens em buscadores através de termos especificados. Defina sua lista de classes.

In [23]:
from imagedl import imagedl
import os
import shutil

# Função para baixar e organizar as imagens
def download_images(keyword, label, n_total=100, folder="data/kirby"):
    final_folder = os.path.join(folder, label)
    os.makedirs(final_folder, exist_ok=True)
    
    # 1. Conta quantas já temos
    count_existing = len([f for f in os.listdir(final_folder) if f.lower().endswith(('.jpg', '.png'))])
    
    # 2. Verifica se precisa baixar mais
    if count_existing >= n_total:
        print(f"[{label}] já possui {count_existing} imagens. Pulando...")
        return

    needed = n_total - count_existing
    print(f"⬇️ [{label}] Tem {count_existing}. Baixando mais {needed}...")

    # Define pasta temporária
    temp_folder = final_folder + "_temp"
    if os.path.exists(temp_folder): shutil.rmtree(temp_folder)

    try:
        # Trocando para DuckDuckGo ou Bing para evitar o erro de timeout do Google/Wikia
        client = imagedl.ImageClient(
            image_source='GoogleImageClient', 
            init_image_client_cfg={'work_dir': temp_folder},
            search_limits=needed + 20, # Pede um pouco a mais de gordura
            num_threadings=4
        )
        
        image_infos = client.search(keyword, search_limits_overrides=needed + 20)
        client.download(image_infos=image_infos)
        
        # 3. Move e Renomeia (incremental)
        new_count = count_existing
        moved = 0
        
        for root, dirs, files in os.walk(temp_folder):
            for file in files:
                if moved >= needed: break # Para se já completou
                
                if file.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')):
                    src = os.path.join(root, file)
                    # Gera nome único para não sobrescrever as antigas (ex: meta_knight_101.jpg)
                    dst = os.path.join(final_folder, f"{label}_{new_count:04d}.jpg")
                    
                    try:
                        shutil.move(src, dst)
                        new_count += 1
                        moved += 1
                    except Exception as e:
                        pass # Ignora erros de arquivo duplicado
                        
        print(f"   Concluído! Total atual de {label}: {new_count}")

    except Exception as e:
        print(f"Erro ao baixar {label}: {e}")
    finally:
        if os.path.exists(temp_folder):
            shutil.rmtree(temp_folder)

In [24]:
dataset_root = 'data/kirby'

# Seus termos de busca
search_terms = {
    "kirby": 'kirby character -waddle -meta -king -bandana -dedede -dee -doo',
    "dee": 'waddle dee character -bandana',
    "bandana_dee": 'bandana waddle dee character',
    "king": 'king dedede character',
    "meta_knight": 'meta knight kirby character',
    "waddle_doo": 'waddle doo kirby character -dee' 
}


if not os.path.exists(dataset_root) or len(os.listdir(dataset_root)) == 0:
    print("downloading")
    for label, term in search_terms.items():
        download_images(term, label, n_total=100, folder=dataset_root)


### Treinamento e Validação

Dividiremos as imagens baixadas nas pastas `train` e `val`. Defina uma porcentagem.

In [25]:
def split_train_val(root_dir, train_ratio=0.8, seed=42):
    random.seed(seed)

    train_dir = root_dir + "_split/train"
    val_dir = root_dir + "_split/val"

    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(val_dir, exist_ok=True)

    for class_name in os.listdir(root_dir):
        class_path = os.path.join(root_dir, class_name)
        if not os.path.isdir(class_path):
            continue

        images = [os.path.join(class_path, f) for f in os.listdir(class_path)]
        images = [f for f in images if os.path.isfile(f)]
        random.shuffle(images)

        n_train = int(len(images) * train_ratio)

        train_class_dir = os.path.join(train_dir, class_name)
        val_class_dir = os.path.join(val_dir, class_name)
        os.makedirs(train_class_dir, exist_ok=True)
        os.makedirs(val_class_dir, exist_ok=True)

        for img in images[:n_train]:
            shutil.copy(img, os.path.join(train_class_dir, os.path.basename(img)))
        for img in images[n_train:]:
            shutil.copy(img, os.path.join(val_class_dir, os.path.basename(img)))

        print(f"{class_name}: {n_train} train, {len(images)-n_train} val")

## Dataset

Implemente um Dataset PyTorch que carregue as imagens baixadas com suas respectivas classes. Aplique data augmentation e carregue em batches.

In [26]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# fazendo ainda

# transform e normalizacao
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])



In [27]:
#dataset
train_dir = 'data/kirby_split/train'
val_dir = 'data/kirby_split/val'
split_train_val(dataset_root)

train_dataset = datasets.ImageFolder(root=train_dir, transform=train_transform)
val_dataset = datasets.ImageFolder(root=val_dir, transform=val_transform)

# DataLoader
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=4,
    pin_memory=True)

class_names = train_dataset.classes
print(class_names)
print(f"Total treino: {len(train_dataset)}")
print(f"Total validação: {len(val_dataset)}")

bandana_dee: 48 train, 12 val
dee: 33 train, 9 val
king: 49 train, 13 val
kirby: 32 train, 8 val
meta_knight: 40 train, 11 val
waddle_doo: 40 train, 11 val
['bandana_dee', 'dee', 'king', 'kirby', 'meta_knight', 'waddle_doo']
Total treino: 242
Total validação: 64


## Definição do Modelo

Defina aqui o modelo que será utilizado, sendo implementação própria ou um modelo pré-treinado. Teste diversas arquiteturas diferentes e verifique qual delas tem melhor desempenho em validação.

In [28]:
# Seu código aqui

## Treinamento

Defina a função de custo e o otimizador do modelo. Em seguida, implemente o código de treinamento e treine-o. Ao final, exiba as curvas de treinamento e validação para a loss e a acurácia.

In [29]:
# Seu código aqui

## Inferência

Calcule algumas métricas como acurácia, matriz de confusão, etc. Em seguida, teste o modelo em novas imagens das classes correspondentes mas de outras fontes (outro buscador, fotos próprias, etc).

In [30]:
# Seu código aqui